# Notebook 03 — Distance-aware training with uncertainty estimation

This notebook trains baseline AMP classifiers using the descriptor matrix generated in **Notebook 02**.

The goal is not hyperparameter optimisation. Instead, this notebook implements a reproducible and interpretable training workflow for the demonstrative case study:

1. Load the curated descriptor dataset.
2. Build a **distance-aware 70/20/10 split** using Euclidean geometry in z-score descriptor space.
3. Train fixed baseline classifiers with **10-fold cross-validation** on the training partition.
4. Estimate prediction uncertainty using a **cross-validation ensemble**.
5. Evaluate train/validation/test performance.
6. Export all trained models, predictions, metrics, and metadata.

SHAP and post hoc explanation analyses are intentionally left for a separate notebook.

In [5]:
# ============================================================
# 1. Imports and configuration
# ============================================================

from __future__ import annotations

import json
import platform
import warnings
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

INPUT_PATH = Path("../outputs/amp_model_ready_descriptors.csv")
FALLBACK_INPUT_PATH = Path("amp_model_ready_descriptors.csv")

OUTPUT_DIR = Path("../outputs_training")
MODEL_DIR = OUTPUT_DIR / "models"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
SPLIT_DIR = OUTPUT_DIR / "splits"

for directory in [OUTPUT_DIR, MODEL_DIR, PREDICTION_DIR, SPLIT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEQUENCE_COL = "sequence"
LABEL_COL = "label"
LABEL_NAME_COL = "label_name"

RANDOM_STATE = 42
N_SPLITS = 10
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.20
TEST_FRACTION = 0.10

MAX_KMEANS_CLUSTERS = 80
MIN_CLUSTER_SIZE = 10

print(f"Run timestamp: {datetime.now().isoformat(timespec='seconds')}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Run timestamp: 2026-05-22T03:15:41
Output directory: /home/david/Desktop/umag_projects/demonstrative_AMR_peptide_discovery/outputs_training


## 2. Load model-ready descriptor dataset

This notebook expects the output of Notebook 02:

```text
outputs/amp_model_ready_descriptors.csv
```

The file should contain at least `sequence`, `label`, and descriptor columns. If the file is in the current directory, the notebook will also try `amp_model_ready_descriptors.csv`.

In [6]:
# ============================================================
# 2. Load model-ready descriptor dataset
# ============================================================

if INPUT_PATH.exists():
    input_path = INPUT_PATH
elif FALLBACK_INPUT_PATH.exists():
    input_path = FALLBACK_INPUT_PATH
else:
    raise FileNotFoundError(
        "Could not find the descriptor dataset. Expected either "
        f"'{INPUT_PATH}' or '{FALLBACK_INPUT_PATH}'. Run Notebook 02 first."
    )

model_df = pd.read_csv(input_path)

required_cols = {SEQUENCE_COL, LABEL_COL}
missing = required_cols.difference(model_df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

model_df = model_df.dropna(subset=[SEQUENCE_COL, LABEL_COL]).copy()
model_df[LABEL_COL] = model_df[LABEL_COL].astype(int)

if LABEL_NAME_COL not in model_df.columns:
    model_df[LABEL_NAME_COL] = np.where(model_df[LABEL_COL] == 1, "AMP", "non-AMP")

metadata_cols = {SEQUENCE_COL, LABEL_COL, LABEL_NAME_COL}
descriptor_cols = [
    col for col in model_df.columns
    if col not in metadata_cols and pd.api.types.is_numeric_dtype(model_df[col])
]

if len(descriptor_cols) == 0:
    raise ValueError("No numeric descriptor columns found.")

model_df[descriptor_cols] = model_df[descriptor_cols].replace([np.inf, -np.inf], np.nan)

print(f"Loaded: {input_path}")
print(f"Dataset shape: {model_df.shape}")
print(f"Number of descriptor columns: {len(descriptor_cols)}")
print("Class distribution:")
print(model_df[LABEL_COL].value_counts().sort_index())

model_df.head()

Loaded: ../outputs/amp_model_ready_descriptors.csv
Dataset shape: (47175, 44)
Number of descriptor columns: 41
Class distribution:
label
0    30661
1    16514
Name: count, dtype: int64


,sequence,label,label_name,length,shannon_entropy,normalized_entropy,unique_residue_fraction,net_charge_pH_7_4,absolute_charge_pH_7_4,isoelectric_point_approx,...,AAC_M,AAC_N,AAC_P,AAC_Q,AAC_R,AAC_S,AAC_T,AAC_V,AAC_W,AAC_Y
0,RIQQIEQKIHHIEQRIQQIEQLLQLTVWGIKQLQARIL,1,AMP,38,3.081120,0.712904,0.315789,2.071371,2.071371,10.57,...,0.000000,0.000000,0.000000,0.263158,0.078947,0.000000,0.026316,0.026316,0.026316,0.000000
1,LHYNWIDCCHYGVSDCC,0,non-AMP,17,3.263933,0.755203,0.647059,-2.379134,2.379134,4.95,...,0.000000,0.058824,0.000000,0.000000,0.000000,0.058824,0.000000,0.058824,0.058824,0.117647
2,WDEDGAKRIPVDVSE,0,non-AMP,15,3.323231,0.768923,0.733333,-3.003945,3.003945,3.81,...,0.000000,0.000000,0.066667,0.000000,0.066667,0.066667,0.000000,0.133333,0.066667,0.000000
3,RQIKKAFRKMA,1,AMP,11,2.663533,0.616283,0.636364,4.992509,4.992509,12.00,...,0.090909,0.000000,0.000000,0.090909,0.181818,0.000000,0.000000,0.000000,0.000000,0.000000
4,MKIKTGARILALSALTTMMFSASALAK,0,non-AMP,27,3.105653,0.718580,0.370370,3.992517,3.992517,11.78,...,0.111111,0.000000,0.000000,0.000000,0.037037,0.111111,0.111111,0.000000,0.000000,0.000000


## 3. Distance-aware 70/20/10 split

The split is based on Euclidean geometry in z-score descriptor space.

To keep the procedure scalable, the notebook first defines geometric strata using MiniBatchKMeans over the standardized descriptor space. Then, samples are allocated inside each class-cluster stratum into train, validation, and test partitions.

This is not intended as a strict homology-aware split. It is a lightweight **descriptor-distance-aware split** suitable for the demonstrative AMP case study.

In [7]:
# ============================================================
# 3. Distance-aware split utilities
# ============================================================

def choose_n_clusters(n_samples: int, max_clusters: int = 80, min_cluster_size: int = 10) -> int:
    """Choose a conservative number of clusters for geometric stratification."""
    if n_samples < min_cluster_size * 2:
        return 2
    return max(2, min(max_clusters, n_samples // min_cluster_size))


def allocate_indices_within_group(
    indices: np.ndarray,
    rng: np.random.Generator,
    train_fraction: float = 0.70,
    validation_fraction: float = 0.20,
    test_fraction: float = 0.10,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Allocate indices into train/validation/test within a local stratum."""
    indices = np.asarray(indices).copy()
    rng.shuffle(indices)

    n = len(indices)
    if n == 1:
        return indices, np.array([], dtype=int), np.array([], dtype=int)
    if n == 2:
        return indices[:1], indices[1:], np.array([], dtype=int)

    n_train = int(round(train_fraction * n))
    n_val = int(round(validation_fraction * n))

    n_train = min(max(n_train, 1), n - 2)
    n_val = min(max(n_val, 1), n - n_train - 1)

    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train + n_val]
    test_idx = indices[n_train + n_val:]

    return train_idx, val_idx, test_idx


def distance_aware_split(
    df: pd.DataFrame,
    descriptor_columns: List[str],
    label_col: str = "label",
    random_state: int = 42,
    max_clusters: int = 80,
    min_cluster_size: int = 10,
) -> pd.DataFrame:
    """
    Generate a distance-aware split using Euclidean clusters in standardized descriptor space.

    The method:
    1. Imputes missing descriptor values using global medians for split construction.
    2. Standardizes descriptors using z-score scaling for geometry estimation.
    3. Runs MiniBatchKMeans using Euclidean distance.
    4. Allocates samples within each (class, geometric_cluster) stratum.
    """
    rng = np.random.default_rng(random_state)

    X_raw = df[descriptor_columns].copy()
    X_imputed = SimpleImputer(strategy="median").fit_transform(X_raw)
    X_scaled = StandardScaler().fit_transform(X_imputed)

    n_clusters = choose_n_clusters(
        n_samples=len(df),
        max_clusters=max_clusters,
        min_cluster_size=min_cluster_size,
    )

    kmeans = MiniBatchKMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        batch_size=2048,
        n_init="auto",
    )
    clusters = kmeans.fit_predict(X_scaled)

    split = pd.Series(index=df.index, dtype="object")
    tmp = pd.DataFrame({
        "index": df.index,
        "label": df[label_col].values,
        "cluster": clusters,
    })

    train_parts, val_parts, test_parts = [], [], []

    for (_, _), group in tmp.groupby(["label", "cluster"], sort=False):
        
        group_indices = group["index"].to_numpy().copy()
        train_idx, val_idx, test_idx = allocate_indices_within_group(
            group_indices,
            rng=rng,
            train_fraction=TRAIN_FRACTION,
            validation_fraction=VALIDATION_FRACTION,
            test_fraction=TEST_FRACTION,
        )
        train_parts.append(train_idx)
        val_parts.append(val_idx)
        test_parts.append(test_idx)

    train_idx = np.concatenate(train_parts) if train_parts else np.array([], dtype=int)
    val_idx = np.concatenate(val_parts) if val_parts else np.array([], dtype=int)
    test_idx = np.concatenate(test_parts) if test_parts else np.array([], dtype=int)

    split.loc[train_idx] = "train"
    split.loc[val_idx] = "validation"
    split.loc[test_idx] = "test"
    split = split.fillna("train")

    split_df = df[[SEQUENCE_COL, LABEL_COL, LABEL_NAME_COL]].copy()
    split_df["split"] = split.values
    split_df["distance_cluster"] = clusters

    return split_df

In [8]:
# ============================================================
# 4. Generate and save distance-aware split
# ============================================================

split_df = distance_aware_split(
    df=model_df,
    descriptor_columns=descriptor_cols,
    label_col=LABEL_COL,
    random_state=RANDOM_STATE,
    max_clusters=MAX_KMEANS_CLUSTERS,
    min_cluster_size=MIN_CLUSTER_SIZE,
)

model_df = model_df.merge(
    split_df[[SEQUENCE_COL, "split", "distance_cluster"]],
    on=SEQUENCE_COL,
    how="left",
)

split_summary = (
    model_df
    .groupby(["split", LABEL_COL])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "non_AMP", 1: "AMP"})
)
split_summary["total"] = split_summary.sum(axis=1)
split_summary["AMP_fraction"] = split_summary.get("AMP", 0) / split_summary["total"]

split_summary.to_csv(SPLIT_DIR / "distance_aware_split_summary.csv")
model_df[[SEQUENCE_COL, LABEL_COL, LABEL_NAME_COL, "split", "distance_cluster"]].to_csv(
    SPLIT_DIR / "distance_aware_split_assignments.csv",
    index=False,
)
model_df.to_csv(SPLIT_DIR / "amp_model_ready_with_distance_aware_split.csv", index=False)

split_summary

label,non_AMP,AMP,total,AMP_fraction
split,,,,
test,3067,1651,4718,0.349936
train,21461,11561,33022,0.350100
validation,6133,3302,9435,0.349974


## 4. Split diagnostics

The notebook quantifies how far validation and test samples are from the nearest training sample in descriptor space. This helps report whether evaluation samples are geometrically close to or distant from the training distribution.

In [9]:
# ============================================================
# 5. Nearest-neighbor split diagnostics in descriptor space
# ============================================================

X_all_raw = model_df[descriptor_cols].replace([np.inf, -np.inf], np.nan)
X_all_imputed = SimpleImputer(strategy="median").fit_transform(X_all_raw)
X_all_scaled = StandardScaler().fit_transform(X_all_imputed)

train_mask = model_df["split"].eq("train").values
val_mask = model_df["split"].eq("validation").values
test_mask = model_df["split"].eq("test").values

nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
nn.fit(X_all_scaled[train_mask])

def nearest_train_distance(mask: np.ndarray) -> np.ndarray:
    if mask.sum() == 0:
        return np.array([])
    distances, _ = nn.kneighbors(X_all_scaled[mask])
    return distances.ravel()

val_dist = nearest_train_distance(val_mask)
test_dist = nearest_train_distance(test_mask)

split_distance_summary = pd.DataFrame({
    "split": ["validation", "test"],
    "n_samples": [len(val_dist), len(test_dist)],
    "mean_nearest_train_distance": [np.mean(val_dist), np.mean(test_dist)],
    "median_nearest_train_distance": [np.median(val_dist), np.median(test_dist)],
    "p90_nearest_train_distance": [np.percentile(val_dist, 90), np.percentile(test_dist, 90)],
})

split_distance_summary.to_csv(SPLIT_DIR / "nearest_train_distance_summary.csv", index=False)
split_distance_summary

,split,n_samples,mean_nearest_train_distance,median_nearest_train_distance,p90_nearest_train_distance
0,validation,9435,2.463425,2.475516,3.521431
1,test,4718,2.479446,2.491641,3.553501


## 5. Fixed models and evaluation metrics

No hyperparameter exploration is performed. The notebook trains a small set of fixed, standard classifiers suitable for descriptor-based AMP prediction.

Uncertainty is estimated from the dispersion of predicted probabilities across the 10 cross-validation models.

In [10]:
# ============================================================
# 6. Fixed model definitions
# ============================================================

models = {
    "logistic_regression": LogisticRegression(
        class_weight="balanced",
        max_iter=5000,
        solver="liblinear",
        random_state=RANDOM_STATE,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "extra_trees": ExtraTreesClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}


def make_pipeline(estimator):
    """Create a leakage-aware preprocessing + model pipeline."""
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", estimator),
    ])


def safe_roc_auc(y_true, y_score) -> float:
    try:
        return roc_auc_score(y_true, y_score)
    except Exception:
        return np.nan


def safe_pr_auc(y_true, y_score) -> float:
    try:
        return average_precision_score(y_true, y_score)
    except Exception:
        return np.nan


def compute_metrics(y_true, y_prob, threshold: float = 0.5) -> Dict[str, float]:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc": safe_roc_auc(y_true, y_prob),
        "pr_auc": safe_pr_auc(y_true, y_prob),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

In [11]:
# ============================================================
# 7. Prepare train / validation / test arrays
# ============================================================

train_df = model_df[model_df["split"] == "train"].copy()
validation_df = model_df[model_df["split"] == "validation"].copy()
test_df = model_df[model_df["split"] == "test"].copy()

X_train = train_df[descriptor_cols]
y_train = train_df[LABEL_COL].values

X_validation = validation_df[descriptor_cols]
y_validation = validation_df[LABEL_COL].values

X_test = test_df[descriptor_cols]
y_test = test_df[LABEL_COL].values

print(f"Train: {X_train.shape}")
print(f"Validation: {X_validation.shape}")
print(f"Test: {X_test.shape}")

Train: (33022, 41)
Validation: (9435, 41)
Test: (4718, 41)


## 6. 10-fold training with uncertainty-aware ensemble predictions

For each classifier:

1. Train 10 fold-specific models on the training set.
2. Use out-of-fold predictions to estimate cross-validation performance.
3. Use the 10 fold models as an ensemble to predict validation and test samples.
4. Estimate uncertainty as the standard deviation of positive-class probabilities across ensemble members.
5. Train one final model on the full training set for deployment/export.

In [13]:
# ============================================================
# 8. Cross-validation ensemble training
# ============================================================

all_metrics = []
trained_artifacts = {}

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

for model_name, estimator in models.items():
    print(f"\n==============================")
    print(f"Training model: {model_name}")
    print(f"==============================")

    fold_models = []
    oof_prob = np.zeros(len(train_df), dtype=float)

    for fold, (fit_idx, holdout_idx) in enumerate(cv.split(X_train, y_train), start=1):
        pipeline = make_pipeline(clone(estimator))
        pipeline.fit(X_train.iloc[fit_idx], y_train[fit_idx])

        holdout_prob = pipeline.predict_proba(X_train.iloc[holdout_idx])[:, 1]
        oof_prob[holdout_idx] = holdout_prob

        fold_models.append(pipeline)
        print(f"Fold {fold:02d} completed")

    def ensemble_predict_proba(X: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        probs = np.column_stack([
            model.predict_proba(X)[:, 1]
            for model in fold_models
        ])
        mean_prob = probs.mean(axis=1)
        uncertainty = probs.std(axis=1)
        return mean_prob, uncertainty

    validation_prob, validation_uncertainty = ensemble_predict_proba(X_validation)
    test_prob, test_uncertainty = ensemble_predict_proba(X_test)
    train_ensemble_prob, train_ensemble_uncertainty = ensemble_predict_proba(X_train)

    final_model = make_pipeline(clone(estimator))
    final_model.fit(X_train, y_train)

    partitions = {
        "train_oof": (y_train, oof_prob),
        "train_ensemble": (y_train, train_ensemble_prob),
        "validation_ensemble": (y_validation, validation_prob),
        "test_ensemble": (y_test, test_prob),
    }

    for partition_name, (y_true, y_prob) in partitions.items():
        metrics = compute_metrics(y_true, y_prob)
        metrics.update({
            "model": model_name,
            "partition": partition_name,
            "n_samples": len(y_true),
        })
        all_metrics.append(metrics)

    train_pred_df = train_df[[SEQUENCE_COL, LABEL_COL, LABEL_NAME_COL, "split"]].copy()
    train_pred_df["probability_amp_oof"] = oof_prob
    train_pred_df["probability_amp_ensemble"] = train_ensemble_prob
    train_pred_df["uncertainty_std"] = train_ensemble_uncertainty
    train_pred_df["prediction"] = (train_pred_df["probability_amp_ensemble"] >= 0.5).astype(int)

    validation_pred_df = validation_df[[SEQUENCE_COL, LABEL_COL, LABEL_NAME_COL, "split"]].copy()
    validation_pred_df["probability_amp_ensemble"] = validation_prob
    validation_pred_df["uncertainty_std"] = validation_uncertainty
    validation_pred_df["prediction"] = (validation_pred_df["probability_amp_ensemble"] >= 0.5).astype(int)

    test_pred_df = test_df[[SEQUENCE_COL, LABEL_COL, LABEL_NAME_COL, "split"]].copy()
    test_pred_df["probability_amp_ensemble"] = test_prob
    test_pred_df["uncertainty_std"] = test_uncertainty
    test_pred_df["prediction"] = (test_pred_df["probability_amp_ensemble"] >= 0.5).astype(int)

    prediction_df = pd.concat([train_pred_df, validation_pred_df, test_pred_df], axis=0)
    prediction_path = PREDICTION_DIR / f"{model_name}_predictions_with_uncertainty.csv"
    prediction_df.to_csv(prediction_path, index=False)

    final_model_path = MODEL_DIR / f"{model_name}_final_model.joblib"
    ensemble_model_path = MODEL_DIR / f"{model_name}_cv_ensemble.joblib"

    joblib.dump(final_model, final_model_path)
    joblib.dump(fold_models, ensemble_model_path)

    trained_artifacts[model_name] = {
        "final_model_path": str(final_model_path),
        "ensemble_model_path": str(ensemble_model_path),
        "prediction_path": str(prediction_path),
    }

    print(f"Saved final model: {final_model_path}")
    print(f"Saved CV ensemble: {ensemble_model_path}")
    print(f"Saved predictions: {prediction_path}")


Training model: logistic_regression
Fold 01 completed
Fold 02 completed
Fold 03 completed
Fold 04 completed
Fold 05 completed
Fold 06 completed
Fold 07 completed
Fold 08 completed
Fold 09 completed
Fold 10 completed
Saved final model: ../outputs_training/models/logistic_regression_final_model.joblib
Saved CV ensemble: ../outputs_training/models/logistic_regression_cv_ensemble.joblib
Saved predictions: ../outputs_training/predictions/logistic_regression_predictions_with_uncertainty.csv

Training model: random_forest
Fold 01 completed
Fold 02 completed
Fold 03 completed
Fold 04 completed
Fold 05 completed
Fold 06 completed
Fold 07 completed
Fold 08 completed
Fold 09 completed
Fold 10 completed
Saved final model: ../outputs_training/models/random_forest_final_model.joblib
Saved CV ensemble: ../outputs_training/models/random_forest_cv_ensemble.joblib
Saved predictions: ../outputs_training/predictions/random_forest_predictions_with_uncertainty.csv

Training model: extra_trees
Fold 01 compl

In [14]:
# ============================================================
# 9. Save and inspect metrics
# ============================================================

metrics_df = pd.DataFrame(all_metrics)
metric_cols_first = ["model", "partition", "n_samples", "mcc", "roc_auc", "pr_auc", "balanced_accuracy", "f1"]
other_cols = [c for c in metrics_df.columns if c not in metric_cols_first]
metrics_df = metrics_df[metric_cols_first + other_cols]

metrics_path = OUTPUT_DIR / "training_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

metrics_df.sort_values(["partition", "mcc"], ascending=[True, False])

,model,partition,n_samples,mcc,roc_auc,pr_auc,balanced_accuracy,f1,accuracy,precision,recall,tn,fp,fn,tp
11,extra_trees,test_ensemble,4718,0.648110,0.910290,0.867803,0.795725,0.739909,0.842942,0.879800,0.638401,2923,144,597,1054
7,random_forest,test_ensemble,4718,0.644196,0.908613,0.862955,0.795143,0.738827,0.841458,0.872218,0.640824,2912,155,593,1058
3,logistic_regression,test_ensemble,4718,0.490666,0.809684,0.737090,0.749784,0.675754,0.763035,0.648303,0.705633,2435,632,486,1165
9,extra_trees,train_ensemble,33022,0.996945,0.999996,0.999991,0.998868,0.998014,0.998607,0.996293,0.999741,21418,43,3,11558
5,random_forest,train_ensemble,33022,0.996939,0.999926,0.999832,0.998509,0.998011,0.998607,0.997838,0.998184,21436,25,21,11540
1,logistic_regression,train_ensemble,33022,0.489757,0.811596,0.738052,0.749035,0.674837,0.762976,0.649241,0.702534,17073,4388,3439,8122
8,extra_trees,train_oof,33022,0.651657,0.906496,0.862384,0.797638,0.742766,0.844407,0.881731,0.641640,20466,995,4143,7418
4,random_forest,train_oof,33022,0.639406,0.904351,0.862268,0.794147,0.737166,0.839531,0.864070,0.642764,20292,1169,4130,7431
0,logistic_regression,train_oof,33022,0.488705,0.810656,0.737075,0.748519,0.674198,0.762461,0.648502,0.702015,17062,4399,3445,8116
10,extra_trees,validation_ensemble,9435,0.649500,0.908280,0.866907,0.796769,0.741416,0.843561,0.879468,0.640824,5843,290,1186,2116


## 7. Select a default model for downstream interpretation

For the next notebook, SHAP and decision-rationale analysis can be performed using the selected model. Here we select the best model by validation MCC. This is **not hyperparameter tuning**; it is only a fixed-model selection step among pre-defined baselines.

In [15]:
# ============================================================
# 10. Select best fixed baseline model by validation MCC
# ============================================================

validation_metrics = metrics_df[metrics_df["partition"] == "validation_ensemble"].copy()
best_row = validation_metrics.sort_values("mcc", ascending=False).iloc[0]
best_model_name = best_row["model"]

best_model_info = {
    "selection_metric": "validation_ensemble_mcc",
    "best_model": best_model_name,
    "best_model_validation_metrics": best_row.to_dict(),
    "artifacts": trained_artifacts[best_model_name],
}

with open(OUTPUT_DIR / "best_model_selection.json", "w") as f:
    json.dump(best_model_info, f, indent=2)

print("Best fixed baseline model:", best_model_name)
best_row

Best fixed baseline model: extra_trees


model                        extra_trees
partition            validation_ensemble
n_samples                           9435
mcc                               0.6495
roc_auc                          0.90828
pr_auc                          0.866907
balanced_accuracy               0.796769
f1                              0.741416
accuracy                        0.843561
precision                       0.879468
recall                          0.640824
tn                                  5843
fp                                   290
fn                                  1186
tp                                  2116
Name: 10, dtype: object

## 8. Export reproducibility metadata

In [17]:
# ============================================================
# 11. Export metadata
# ============================================================

metadata = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "input_path": str(input_path),
    "n_samples": int(len(model_df)),
    "n_descriptors": int(len(descriptor_cols)),
    "descriptor_columns": descriptor_cols,
    "label_column": LABEL_COL,
    "sequence_column": SEQUENCE_COL,
    "split_strategy": "descriptor-distance-aware geometric stratification with Euclidean MiniBatchKMeans clusters",
    "split_fractions": {
        "train": TRAIN_FRACTION,
        "validation": VALIDATION_FRACTION,
        "test": TEST_FRACTION,
    },
    "distance_space": "z-score standardized descriptor space",
    "n_cv_splits": N_SPLITS,
    "uncertainty_strategy": "standard deviation of positive-class probabilities across 10 cross-validation ensemble members",
    "models": list(models.keys()),
    "random_state": RANDOM_STATE,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "artifacts": trained_artifacts,
    "best_model": best_model_info,
}

metadata_path = OUTPUT_DIR / "training_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved metadata:", metadata_path)
print("Saved files:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print("-", path)

Saved metadata: ../outputs_training/training_metadata.json
Saved files:
- ../outputs_training/best_model_selection.json
- ../outputs_training/models/extra_trees_cv_ensemble.joblib
- ../outputs_training/models/extra_trees_final_model.joblib
- ../outputs_training/models/logistic_regression_cv_ensemble.joblib
- ../outputs_training/models/logistic_regression_final_model.joblib
- ../outputs_training/models/random_forest_cv_ensemble.joblib
- ../outputs_training/models/random_forest_final_model.joblib
- ../outputs_training/predictions/extra_trees_predictions_with_uncertainty.csv
- ../outputs_training/predictions/logistic_regression_predictions_with_uncertainty.csv
- ../outputs_training/predictions/random_forest_predictions_with_uncertainty.csv
- ../outputs_training/splits/amp_model_ready_with_distance_aware_split.csv
- ../outputs_training/splits/distance_aware_split_assignments.csv
- ../outputs_training/splits/distance_aware_split_summary.csv
- ../outputs_training/splits/nearest_train_distanc

## 9. Notes for the manuscript

This notebook supports the demonstrative case study by producing:

- A descriptor-based **distance-aware 70/20/10 split**.
- Fixed baseline classifiers trained without hyperparameter search.
- 10-fold cross-validation performance on the training set.
- Ensemble-based uncertainty estimates for candidate prioritisation.
- Exported final models and CV ensembles for downstream SHAP and scoring notebooks.

The next notebook should focus exclusively on:

1. SHAP-based global and local explanation.
2. Extraction of interpretable design rules.
3. Rule-aware candidate ranking using probability, uncertainty, and physicochemical filters.